# Guía 03 / Proyecto Equipo 8 HRTech: Procesamiento e Ingesta de Datos a Parquet
**Proyecto:** Equipo 8 HRTech People Analytics  
**Descripción:** Carga del dataset limpio de la Guía 02, conversión e ingesta a formato optimizado Parquet y cálculo de KPIs analíticos de People Analytics.

In [1]:
# 1. Importación de librerías esenciales
from pathlib import Path
import pandas as pd
import time

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


In [2]:
# 2. Definición de Rutas Relativas e Ingesta del Dataset Limpio (Guía 02)
BASE_DIR = Path("../")
DATOS_DIR = BASE_DIR / "datos"

ruta_csv = DATOS_DIR / "dataset_limpio_guia02.csv"

print(f"Cargando dataset desde: {ruta_csv}")
start_time = time.time()
df_csv = pd.read_csv(ruta_csv)
tiempo_carga_csv = time.time() - start_time

print(f"✓ Registros cargados: {len(df_csv):,}")
print(f"✓ Columnas detectadas: {len(df_csv.columns)}")
print(f"✓ Tiempo de carga CSV: {tiempo_carga_csv:.4f} segundos\n")
print("Esquema y tipos de datos:")
print(df_csv.dtypes)

Cargando dataset desde: ..\datos\dataset_limpio_guia02.csv
✓ Registros cargados: 160
✓ Columnas detectadas: 42
✓ Tiempo de carga CSV: 0.0129 segundos

Esquema y tipos de datos:
registro_id                             int64
empleado_id                               str
periodo                                   str
fecha_evaluacion                          str
productividad_pct                     float64
calidad_trabajo_pct                   float64
cumplimiento_objetivos_pct            float64
asistencia_pct                        float64
horas_capacitacion                    float64
horas_extra                             int64
ausencias_dias                        float64
puntaje_desempeno                     float64
nivel_desempeno                           str
nombre_completo                           str
genero                                    str
edad                                  float64
fecha_ingreso                             str
salario_mensual                       flo

In [3]:
# 3. Conversión y Exportación a Formato Parquet (Compresión Snappy)
ruta_parquet = DATOS_DIR / "dataset_limpio_guia02.parquet"

start_time = time.time()
df_csv.to_parquet(ruta_parquet, engine="pyarrow", compression="snappy", index=False)
tiempo_escritura_parquet = time.time() - start_time

print(f"✓ Dataset exportado exitosamente a Parquet en: {ruta_parquet}")
print(f"✓ Tiempo de escritura en Parquet: {tiempo_escritura_parquet:.4f} segundos")

✓ Dataset exportado exitosamente a Parquet en: ..\datos\dataset_limpio_guia02.parquet
✓ Tiempo de escritura en Parquet: 0.0396 segundos


In [4]:
# 4. Lectura Optimizada desde Archivo Parquet
start_time = time.time()
df_parquet = pd.read_parquet(ruta_parquet, engine="pyarrow")
tiempo_lectura_parquet = time.time() - start_time

print(f"✓ Tiempo de lectura desde Parquet: {tiempo_lectura_parquet:.4f} segundos\n")
print("Muestra de los primeros 5 registros:")
print(df_parquet.head(5))

✓ Tiempo de lectura desde Parquet: 0.0393 segundos

Muestra de los primeros 5 registros:
   registro_id empleado_id  periodo fecha_evaluacion  productividad_pct  \
0            1      EMP001  2025-Q3       2025-09-30               76.2   
1            2      EMP001  2025-Q4       2025-12-31               56.2   
2            3      EMP001  2026-Q1       2026-03-31               75.1   
3            4      EMP001  2026-Q2       2026-06-30               71.6   
4            5      EMP002  2025-Q3       2025-09-30               76.5   

   calidad_trabajo_pct  cumplimiento_objetivos_pct  asistencia_pct  \
0                 72.1                        68.6            97.6   
1                 76.7                        71.5            94.1   
2                 67.9                        76.0            92.8   
3                 79.8                        78.1            93.3   
4                 80.2                        86.3            98.7   

   horas_capacitacion  horas_extra  ...

In [5]:
# 5. Análisis y Métricas de People Analytics (Equipo 8 HRTech)
cols = [c.lower() for c in df_parquet.columns]
print("Columnas disponibles en el dataset:", list(df_parquet.columns))

# Búsqueda dinámica de la columna de departamento / área laboral
col_dept = None
for posible_nombre in ["department", "departamento", "area", "dept"]:
    if posible_nombre in cols:
        col_dept = df_parquet.columns[cols.index(posible_nombre)]
        break

if col_dept:
    print(f"\nGenerando resumen analítico agrupado por: '{col_dept}'\n")
    kpis_dept = df_parquet.groupby(col_dept).size().reset_index(name="total_empleados")
    kpis_dept = kpis_dept.sort_values(by="total_empleados", ascending=False)
    
    print("=== Distribución de Empleados por Departamento ===")
    print(kpis_dept.to_string(index=False))
    
    # Exportar KPIs procesados a Parquet
    ruta_kpis = DATOS_DIR / "kpis_hrtech_departamentos.parquet"
    kpis_dept.to_parquet(ruta_kpis, engine="pyarrow", index=False)
    print(f"\n✓ KPIs exportados exitosamente a: {ruta_kpis}")
else:
    print("\nResumen estadístico de las variables:")
    print(df_parquet.describe())

Columnas disponibles en el dataset: ['registro_id', 'empleado_id', 'periodo', 'fecha_evaluacion', 'productividad_pct', 'calidad_trabajo_pct', 'cumplimiento_objetivos_pct', 'asistencia_pct', 'horas_capacitacion', 'horas_extra', 'ausencias_dias', 'puntaje_desempeno', 'nivel_desempeno', 'nombre_completo', 'genero', 'edad', 'fecha_ingreso', 'salario_mensual', 'modalidad', 'estado_laboral', 'fecha_salida', 'puesto', 'nivel_puesto', 'departamento', 'ubicacion_departamento', 'encuesta_id', 'fecha_encuesta', 'canal', 'comentario_abierto', 'sentimiento_declarado', 'anonimizada', 'respuestas_satisfaccion_laboral', 'respuestas_relacion_con_jefatura', 'respuestas_reconocimiento', 'respuestas_balance_vida_trabajo', 'respuestas_intencion_de_permanecer', 'indice_satisfaccion_promedio', 'antiguedad', 'capacitaciones_recibidas', 'nivel_desempeno_calculado', 'riesgo_rotacion', 'estado_colaborador']

Generando resumen analítico agrupado por: 'departamento'

=== Distribución de Empleados por Departamento 